In [1]:
# Instalação das dependências incluindo o suporte local a Hugging Face
!pip install -q -U \
    llama-index \
    llama-index-embeddings-huggingface \
    llama-index-llms-huggingface \
    transformers \
    accelerate \
    bitsandbytes \
    sentence-transformers \
    wget

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
!pip install -q pymupdf pypdf llama-index-readers-file

In [3]:
import torch

print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM:", torch.cuda.get_device_properties(0).total_memory / 1024**3, "GB")

CUDA: True
GPU: Tesla T4
VRAM: 14.56317138671875 GB


In [5]:
import wget
import os

url = "https://raw.githubusercontent.com/buenofgustavo/projeto-sistemas-multiagentes/main/dados/entradas/v2/explicacoes.jsonl"
output_dir = "data"

# Cria a pasta 'data' se ela não existir
os.makedirs(output_dir, exist_ok=True)

# Faz o download do arquivo
file_path = os.path.join(output_dir, "explicacoes.jsonl")
if not os.path.exists(file_path):
    print("Baixando o arquivo...")
    wget.download(url, file_path)
    print("\n✅ Download concluído!")
else:
    print("✅ O arquivo já existe na pasta.")

Baixando o arquivo...


HTTPError: HTTP Error 404: Not Found

In [ ]:
import json
from llama_index.core import Document

documents = []

with open(file_path, "r", encoding="utf-8") as arquivo:
    for linha in arquivo:
        if not linha.strip():
            continue

        registro = json.loads(linha)

        # Guarda os demais campos como metadados
        metadados = {
            chave: valor
            for chave, valor in registro.items()
            if chave != "explicacao_negocio"
        }

        documento = Document(
            id_=registro["id"],
            text=registro["explicacao_negocio"],
            metadata=metadados,
            # Usa apenas a explicação para gerar os embeddings
            excluded_embed_metadata_keys=list(metadados),
            # Código disponível nos metadados para consulta explícita
            excluded_llm_metadata_keys=["parametrizacao_gobr"],
        )

        documents.append(documento)

print(f"✅ Total de documentos carregados: {len(documents)}")

In [ ]:
from llama_index.core import Settings
from llama_index.llms.huggingface import HuggingFaceLLM
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# LLM LOCAL

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

Settings.llm = HuggingFaceLLM(
    model_name=MODEL_NAME,
    tokenizer_name=MODEL_NAME,

    context_window=8192,
    max_new_tokens=1024,

    device_map="auto",

    model_kwargs={
        "dtype": torch.float16,
    },

    generate_kwargs={
        "temperature": 0.1,
        "do_sample": False,
    },
)

# EMBEDDING LOCAL

Settings.embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-m3",
    device="cuda",
    embed_batch_size=16,
)


print("✅ LLM local configurado")
print("   Qwen2.5-3B-Instruct 4-bit")

print("✅ Embedding local configurado")
print("   BAAI/bge-m3")

In [ ]:
from llama_index.core import VectorStoreIndex
from llama_index.core.node_parser import SentenceSplitter

# 1. Configura o fatiador globalmente
# chunk_size: tamanho em tokens/palavras de cada trecho
# chunk_overlap: sobreposição para evitar que o contexto se perca entre quebras
Settings.text_splitter = SentenceSplitter(
    chunk_size=512,      # Tamanho recomendado para modelos padrão de embedding
    chunk_overlap=64     # Sobreposição de 64 tokens entre chunks consecutivos
)

print("Criando o índice de vetores na memória RAM...")
# O processo de indexação calcula os vetores e os mantém na memória
index = VectorStoreIndex.from_documents(documents)

print("✅ Índice criado com sucesso e mantido na memória RAM!")

In [ ]:
# Transforma o índice em uma engine de busca e resposta
query_engine = index.as_query_engine(similarity_top_k=3, response_mode="compact",)

# Exemplo de pergunta baseada no conteúdo do PDF baixado
pergunta = """
"""

print(f"❓ Pergunta: {pergunta}\n")
print("🔍 Buscando no documento e gerando resposta com o modelo local:\n")

resposta = query_engine.query(pergunta)

print("🤖 Resposta do Modelo:")
print(resposta.response)

In [ ]:
# Visualização das fontes/trechos recuperados na busca
print("📄 Trechos do documento (Nodes) utilizados para construir a resposta:\n")

for i, source_node in enumerate(resposta.source_nodes, 1):
    print(f"--- Fonte {i} (Score de Similaridade: {source_node.score:.4f}) ---")
    # Exibe os primeiros 300 caracteres do texto recuperado
    print(source_node.node.get_content()[:300].replace("\n", " "))
    print("\n" + "="*100 + "\n")